# V1-S15 — F2 dual-novelty TRENDS (enrichment of the finding that HOLDS)

Finding **F2** (the dual-novelty 2×2 archetype construct) already **holds**: Gate G3
(PROCEED, signed 2026-05-31) established that the semantic and structural novelty axes are
independent (max |ρ| = 0.18 ≪ 0.4) and affirmed the *surprising-to-expert* impact-by-archetype
inversion. This notebook does **NOT** re-gate F2 — it **enriches** the F2 narrative with three
new descriptive views and re-confirms (and quantifies) the inversion with a single omnibus test.
Since F2 is the one finding that holds in the downscoped paper, its enrichment carries the story.

We use the **canonical** archetype column `arch_mean_cd5` (the primary `sem_nov_mean × cd5`
pairing per G3) and the locked, age-fair impact metric `cited_by_pctile_within_year` (the
within-year citation percentile, 0–1). Null-archetype rows (a paper missing either novelty
axis — earliest-in-field or no CD citers) are **excluded** from the 2×2, never imputed.

New analyses (none re-gate F2):
1. **Archetype distribution BY YEAR** — the four `arch_mean_cd5` shares over time (G3's
   temporal drift: conventional-disruptive rising, novel-consolidating falling).
2. **Archetype distribution BY TOPIC** — per-topic archetype shares for the largest leaf topics
   (a compact heatmap) + each topic's dominant archetype.
3. **Archetype × impact (the headline enrichment)** — `kruskal_archetype_impact(...)` on
   `cited_by_pctile_within_year`; reports H, p, ε² and the per-archetype median percentile, and
   confirms the disruptive-novel-lowest / incremental-highest inversion.

**$0, CPU-only, no network/GPU/DeepSeek.** All statistics live in the pure module
`scifield.findings.impact`; this notebook only does the parquet I/O and figure.

## 1. Setup

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

# Repo-root sniff — notebook runs from notebooks/, code lives one dir up.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

DATA = repo_root / "data" / "v1"
FIGURES_DIR = repo_root / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DPI = 120

from scifield.findings.impact import kruskal_archetype_impact  # noqa: E402
from scifield.novelty.archetypes import QUADRANT_LABELS  # noqa: E402
from scifield.repro import record_run  # noqa: E402

# Canonical column choices (locked per Gate G3).
ARCH_COL = "arch_mean_cd5"  # primary sem_nov_mean × cd5 pairing
IMPACT_COL = "cited_by_pctile_within_year"  # age-fair within-year citation percentile

# Fixed archetype order, low-impact -> high-impact per G3 (for stable colors/legends).
ARCH_ORDER = [
    "disruptive-novel",  # HH
    "novel-consolidating",  # HL
    "conventional-disruptive",  # LH
    "incremental",  # LL
]
ARCH_COLORS = {
    "disruptive-novel": "#ea4335",
    "novel-consolidating": "#fbbc04",
    "conventional-disruptive": "#4285f4",
    "incremental": "#34a853",
}
assert set(ARCH_ORDER) == set(QUADRANT_LABELS.values()), "archetype label set drift vs module"


def _load_parquet(path: Path, columns=None) -> pd.DataFrame | None:
    """Defensive parquet read — return None (with a message) if absent."""
    if not path.exists():
        print(f"MISSING artifact: {path} — cannot run the analysis.")
        return None
    return pd.read_parquet(path, columns=columns)


print("matplotlib:", matplotlib.__version__, "| backend:", matplotlib.get_backend())
print("DATA       :", DATA)
print("FIGURES_DIR:", FIGURES_DIR)

matplotlib: 3.10.9 | backend: Agg
DATA       : /Users/samersalman/Desktop/SciField/data/v1
FIGURES_DIR: /Users/samersalman/Desktop/SciField/docs/figures


## 2. Load `archetypes.parquet`, exclude null-archetype rows

The G3 artifact `data/v1/archetypes.parquet` carries one row per corpus paper with the four
archetype label columns and the locked impact metric. We read the columns we need, then split
into the **2×2 universe** (non-null `arch_mean_cd5`) for the by-year / by-topic / impact views.
Null-archetype rows are reported but excluded (documented, not silently dropped — G3 §0).

In [2]:
ARCHETYPES = DATA / "archetypes.parquet"
cols = ["pmid", "year", "topic_id", ARCH_COL, IMPACT_COL]
adf = _load_parquet(ARCHETYPES, columns=cols)
assert adf is not None, "archetypes.parquet missing — cannot proceed."

n_total = len(adf)
n_null_arch = int(adf[ARCH_COL].isna().sum())
# The 2×2 universe: rows with a non-null canonical archetype label.
df = adf[adf[ARCH_COL].notna()].copy()
n_labeled = len(df)

# Pin the archetype column to the fixed low->high-impact category order.
df[ARCH_COL] = pd.Categorical(df[ARCH_COL], categories=ARCH_ORDER, ordered=True)

print(f"total papers           = {n_total:,}")
print(
    f"null-archetype (excl.) = {n_null_arch:,}  "
    f"(earliest-in-field / no CD citers — excluded from the 2×2, not imputed)"
)
print(f"2×2 universe (labeled)  = {n_labeled:,}")
print(f"year span               = {int(df['year'].min())}–{int(df['year'].max())}")
print("\nper-archetype counts (canonical arch_mean_cd5):")
print(df[ARCH_COL].value_counts().reindex(ARCH_ORDER).to_string())

total papers           = 89,230
null-archetype (excl.) = 7,492  (earliest-in-field / no CD citers — excluded from the 2×2, not imputed)
2×2 universe (labeled)  = 81,738
year span               = 1996–2026

per-archetype counts (canonical arch_mean_cd5):
arch_mean_cd5
disruptive-novel           22662
novel-consolidating        18210
conventional-disruptive    19474
incremental                21392


## 3. Analysis 1 — archetype distribution BY YEAR

The four `arch_mean_cd5` shares per publication year. G3 reported a temporal drift from ≤2000 to
≥2020: **conventional-disruptive rising (+0.083)** and **novel-consolidating falling
(−0.055 (G3: −0.056))**. We compute per-year shares and the same two-epoch endpoints, restricting
the endpoint contrast to years with a non-trivial denominator (the early-year CD coverage caveat
from G3 §2 still applies — pre-2000 shares ride a thinner CD denominator).

In [3]:
# Per-(year, archetype) share within each year.
by_year_n = df.groupby(["year", ARCH_COL], observed=True).size().rename("n").reset_index()
year_tot = by_year_n.groupby("year")["n"].transform("sum")
by_year_n["share"] = by_year_n["n"] / year_tot

# Wide year × archetype share matrix (rows = years, cols = the 4 archetypes).
share_by_year = (
    by_year_n.pivot(index="year", columns=ARCH_COL, values="share")
    .reindex(columns=ARCH_ORDER)
    .sort_index()
)
year_counts = df.groupby("year", observed=True).size()
print(
    f"per-year paper counts (labeled): min {int(year_counts.min())}, "
    f"max {int(year_counts.max())}"
)

# Two-epoch endpoints (G3 used ≤2000 vs ≥2020).
early = df[df["year"] <= 2000]
late = df[df["year"] >= 2020]
drift = {}
print("\narchetype shares: ≤2000 vs ≥2020 (G3 endpoints)")
for a in ARCH_ORDER:
    e_share = float((early[ARCH_COL] == a).mean())
    l_share = float((late[ARCH_COL] == a).mean())
    drift[a] = {"share_le2000": e_share, "share_ge2020": l_share, "delta": l_share - e_share}
    print(f"  {a:<26} ≤2000={e_share:.3f}  ≥2020={l_share:.3f}  Δ={l_share - e_share:+.3f}")

CD_DRIFT = drift["conventional-disruptive"]["delta"]
NC_DRIFT = drift["novel-consolidating"]["delta"]
print(
    f"\nG3 temporal-drift reconfirm: conventional-disruptive Δ={CD_DRIFT:+.3f} (rising), "
    f"novel-consolidating Δ={NC_DRIFT:+.3f} (falling)"
)

per-year paper counts (labeled): min 333, max 3281

archetype shares: ≤2000 vs ≥2020 (G3 endpoints)
  disruptive-novel           ≤2000=0.326  ≥2020=0.285  Δ=-0.040
  novel-consolidating        ≤2000=0.241  ≥2020=0.185  Δ=-0.055
  conventional-disruptive    ≤2000=0.206  ≥2020=0.289  Δ=+0.083
  incremental                ≤2000=0.228  ≥2020=0.240  Δ=+0.013

G3 temporal-drift reconfirm: conventional-disruptive Δ=+0.083 (rising), novel-consolidating Δ=-0.055 (falling)


## 4. Analysis 2 — archetype distribution BY TOPIC

Per-topic archetype shares for the largest leaf topics. The noise cluster (`topic_id == -1`) is
excluded — it is not a coherent topic. We show a compact heatmap of the top-N topics by labeled
paper count (readable labels from the BERTopic `top_words`, mirroring notebook 11), plus each
topic's **dominant** archetype. This surfaces cross-topic heterogeneity without re-gating anything.

In [4]:
TOPN_TOPICS = 18  # compact heatmap height

# Readable topic labels from the hierarchy's top_words (first 3 words) — optional.
th = _load_parquet(DATA / "topic_hierarchy.parquet", columns=["topic_id", "top_words"])
TOPIC_WORDS: dict[int, str] = {}
if th is not None:
    for row in th.itertuples():
        words = list(row.top_words)
        TOPIC_WORDS[int(row.topic_id)] = ", ".join(str(w) for w in words[:3])

# Leaf topics only (drop the -1 noise cluster), ranked by labeled paper count.
leaf = df[df["topic_id"] != -1].copy()
topic_counts = leaf.groupby("topic_id", observed=True).size().sort_values(ascending=False)
top_topics = topic_counts.head(TOPN_TOPICS).index.tolist()

# Per-topic archetype share matrix (topics × 4 archetypes) for the top-N topics.
sub = leaf[leaf["topic_id"].isin(top_topics)]
topic_share = sub.groupby(["topic_id", ARCH_COL], observed=True).size().rename("n").reset_index()
topic_tot = topic_share.groupby("topic_id")["n"].transform("sum")
topic_share["share"] = topic_share["n"] / topic_tot
share_by_topic = (
    topic_share.pivot(index="topic_id", columns=ARCH_COL, values="share")
    .reindex(columns=ARCH_ORDER)
    .reindex(index=top_topics)
)

# Dominant archetype per topic (argmax share) across ALL leaf topics.
all_topic_share = (
    leaf.groupby(["topic_id", ARCH_COL], observed=True).size().rename("n").reset_index()
)
all_tot = all_topic_share.groupby("topic_id")["n"].transform("sum")
all_topic_share["share"] = all_topic_share["n"] / all_tot
dom = (
    all_topic_share.sort_values("share", ascending=False)
    .groupby("topic_id")
    .head(1)
    .set_index("topic_id")[ARCH_COL]
)
dom_counts = dom.value_counts().reindex(ARCH_ORDER).fillna(0).astype(int)
print(f"leaf topics (excl. -1) = {leaf['topic_id'].nunique()}")
print(f"dominant-archetype tally across all {len(dom)} leaf topics:")
for a in ARCH_ORDER:
    print(f"  {a:<26} dominant in {int(dom_counts[a])} topics")

print(f"\ntop {TOPN_TOPICS} topics by labeled count (with dominant archetype):")
for t in top_topics[:8]:
    lbl = TOPIC_WORDS.get(int(t), f"topic {t}")
    print(f"  topic {int(t):>3} (n={int(topic_counts[t]):>4})  dom={dom.get(t, 'NA'):<24} {lbl}")

leaf topics (excl. -1) = 149
dominant-archetype tally across all 149 leaf topics:
  disruptive-novel           dominant in 30 topics
  novel-consolidating        dominant in 5 topics
  conventional-disruptive    dominant in 48 topics
  incremental                dominant in 66 topics

top 18 topics by labeled count (with dominant archetype):
  topic   0 (n=2397)  dom=incremental              acetabular, hip, femoral
  topic   1 (n=2216)  dom=disruptive-novel         infection, pji, periprosthetic
  topic   2 (n=2174)  dom=incremental              acl, cruciate, ligament
  topic   3 (n=1777)  dom=incremental              liver, resection, hepatic
  topic   6 (n=1352)  dom=disruptive-novel         breast, breast cancer, cancer
  topic   5 (n=1339)  dom=incremental              scoliosis, idiopathic scoliosis, idiopathic
  topic   7 (n=1330)  dom=conventional-disruptive  arthroplasty, tka, total
  topic   9 (n=1320)  dom=incremental              knee, knee arthroplasty, total knee


## 5. Analysis 3 — archetype × impact (the headline enrichment)

The genuinely surprising G3 result (Row 2, designated surprising-to-expert): the most
novel-and-disruptive papers sit **LOWER** on age-fair within-year citation impact than incremental
ones. We quantify it with one omnibus **Kruskal-Wallis** test across the four archetypes — a
nonparametric ANOVA on ranks (the impact metric is a bounded within-year percentile, not
Gaussian) — plus the rank-based effect size **ε² = (H − k + 1)/(n − k)**. The pure helper
`kruskal_archetype_impact` does the drop-nulls / group / test / effect-size; this cell just
calls it and prints the numbers.

**Effect-size note:** the test is hugely significant (H≈1938, p<1e-300), but ε²≈0.024 is a
*small* effect. So the archetype **ordering** (disruptive-novel lowest → incremental highest) is
highly reliable, while the **impact gap is modest** — archetype explains only ~2.4% of the
within-year-impact rank variance.

In [5]:
kw = kruskal_archetype_impact(df, archetype_col=ARCH_COL, impact_col=IMPACT_COL)

medians = {a: kw["per_group"][a]["median"] for a in ARCH_ORDER}
ns = {a: kw["per_group"][a]["n"] for a in ARCH_ORDER}

print("=== Kruskal-Wallis: cited_by_pctile_within_year across arch_mean_cd5 ===")
# p underflows to 0.0 in float; report it as such but keep the statistic exact.
p_disp = "< 1e-300 (float underflow)" if kw["p_value"] == 0.0 else f"{kw['p_value']:.3e}"
print(f"H           = {kw['H']:.4f}")
print(f"p_value     = {p_disp}")
print(f"k (groups)  = {kw['k']}")
print(f"n           = {kw['n']:,}")
print(f"epsilon^2   = {kw['epsilon_squared']:.5f}  (rank-based effect size)")
print("\nper-archetype median within-year citation percentile (n):")
for a in ARCH_ORDER:
    print(f"  {a:<26} median={medians[a]:.4f}  (n={ns[a]:,})")

# The inversion test: disruptive-novel lowest, incremental highest, and the
# medians monotonically increase across the G3 order.
ordered_medians = [medians[a] for a in ARCH_ORDER]
dn_lowest = medians["disruptive-novel"] == min(medians.values())
inc_highest = medians["incremental"] == max(medians.values())
monotone = ordered_medians == sorted(ordered_medians)
INVERSION_RECONFIRMED = bool(dn_lowest and inc_highest and monotone)
print(f"\ndisruptive-novel is LOWEST  : {dn_lowest}  (median {medians['disruptive-novel']:.3f})")
print(f"incremental is HIGHEST       : {inc_highest}  (median {medians['incremental']:.3f})")
print(f"medians monotone dn<nc<cd<inc : {monotone}")
print(
    f"\n>>> G3 impact-by-archetype INVERSION RECONFIRMED = {INVERSION_RECONFIRMED} "
    f"(significant: ε²={kw['epsilon_squared']:.4f}, H={kw['H']:.1f})"
)
# Effect-size caveat: hugely significant but a SMALL effect. The archetype ORDERING is
# highly reliable, but the impact gap is modest — archetype explains only ~2.4% of the
# within-year-impact rank variance.
eps_pct = kw["epsilon_squared"] * 100
print(
    f">>> EFFECT SIZE is SMALL: ε²={kw['epsilon_squared']:.4f} (~{eps_pct:.1f}% of "
    "within-year-impact rank variance) — ordering reliable, impact gap modest."
)

=== Kruskal-Wallis: cited_by_pctile_within_year across arch_mean_cd5 ===
H           = 1938.5837
p_value     = < 1e-300 (float underflow)
k (groups)  = 4
n           = 81,738
epsilon^2   = 0.02368  (rank-based effect size)

per-archetype median within-year citation percentile (n):
  disruptive-novel           median=0.4329  (n=22,662)
  novel-consolidating        median=0.5023  (n=18,210)
  conventional-disruptive    median=0.5234  (n=19,474)
  incremental                median=0.6069  (n=21,392)

disruptive-novel is LOWEST  : True  (median 0.433)
incremental is HIGHEST       : True  (median 0.607)
medians monotone dn<nc<cd<inc : True

>>> G3 impact-by-archetype INVERSION RECONFIRMED = True (significant: ε²=0.0237, H=1938.6)
>>> EFFECT SIZE is SMALL: ε²=0.0237 (~2.4% of within-year-impact rank variance) — ordering reliable, impact gap modest.


## 6. Figure — `docs/figures/F2_dual_novelty_trends.png`

One multi-panel figure: **(a)** the by-year archetype shares (stacked area — the G3 temporal
drift), **(b)** the by-topic heatmap (within-topic archetype share for the top-`TOPN_TOPICS` leaf
topics; rows = topic, cols = the 4 archetypes), and **(c)** the archetype × impact distribution
(box of `cited_by_pctile_within_year` by archetype, annotated with the Kruskal-Wallis H, p, ε²
and the per-archetype median). Saved at DPI=120; asserted < 1 MB.

In [6]:
fig, (ax_year, ax_topic, ax_imp) = plt.subplots(1, 3, figsize=(19, 5.8))

# --- (a) archetype shares BY YEAR (stacked area) ---
yrs = share_by_year.index.to_numpy()
stack_vals = [share_by_year[a].to_numpy() for a in ARCH_ORDER]
ax_year.stackplot(
    yrs,
    *stack_vals,
    labels=ARCH_ORDER,
    colors=[ARCH_COLORS[a] for a in ARCH_ORDER],
    alpha=0.9,
)
ax_year.set_xlim(int(yrs.min()), int(yrs.max()))
ax_year.set_ylim(0, 1)
ax_year.set_xlabel("publication year")
ax_year.set_ylabel("archetype share")
ax_year.set_title(
    "(a) Archetype share by year (arch_mean_cd5)\n"
    f"≤2000→≥2020: conventional-disruptive {CD_DRIFT:+.3f}, "
    f"novel-consolidating {NC_DRIFT:+.3f}",
    fontsize=9.5,
)
ax_year.legend(loc="lower center", ncol=2, fontsize=7, framealpha=0.9)

# --- (b) archetype shares BY TOPIC (heatmap of the top-N leaf topics) ---
# rows = top-N topics (largest first), cols = the 4 archetypes, color = share.
heat = share_by_topic.reindex(index=top_topics, columns=ARCH_ORDER).to_numpy()
im = ax_topic.imshow(heat, aspect="auto", cmap="viridis", vmin=0, vmax=heat.max())
ax_topic.set_xticks(range(len(ARCH_ORDER)))
ax_topic.set_xticklabels([a.replace("-", "-\n") for a in ARCH_ORDER], fontsize=7.5)
# Readable y labels from BERTopic top_words (fall back to topic_id).
ylabels = [f"{int(t)}: {TOPIC_WORDS.get(int(t), '')}".rstrip(": ") for t in top_topics]
ax_topic.set_yticks(range(len(top_topics)))
ax_topic.set_yticklabels(ylabels, fontsize=6.5)
ax_topic.set_title(
    f"(b) Archetype share by topic — top {TOPN_TOPICS} leaf topics\n"
    "(rows = topic, cols = archetype, color = within-topic share)",
    fontsize=9.5,
)
cbar = fig.colorbar(im, ax=ax_topic, fraction=0.046, pad=0.04)
cbar.set_label("within-topic archetype share", fontsize=8)
cbar.ax.tick_params(labelsize=7)

# --- (c) archetype × impact (box of within-year citation percentile) ---
box_data = [df.loc[df[ARCH_COL] == a, IMPACT_COL].dropna().to_numpy() for a in ARCH_ORDER]
bp = ax_imp.boxplot(
    box_data,
    labels=[a.replace("-", "-\n") for a in ARCH_ORDER],
    showfliers=False,
    patch_artist=True,
    medianprops=dict(color="black", lw=1.6),
    widths=0.6,
)
for patch, a in zip(bp["boxes"], ARCH_ORDER, strict=False):
    patch.set_facecolor(ARCH_COLORS[a])
    patch.set_alpha(0.85)
# Annotate each median value above the box.
for i, a in enumerate(ARCH_ORDER, start=1):
    ax_imp.text(
        i, medians[a] + 0.02, f"{medians[a]:.3f}", ha="center", fontsize=8, fontweight="bold"
    )
ax_imp.set_ylabel("within-year citation percentile")
ax_imp.set_ylim(0, 1)
ax_imp.tick_params(axis="x", labelsize=8)
p_lbl = "p<1e-300" if kw["p_value"] == 0.0 else f"p={kw['p_value']:.1e}"
ax_imp.set_title(
    "(c) Impact by archetype — the G3 inversion, quantified\n"
    f"Kruskal-Wallis H={kw['H']:.0f}, {p_lbl}, ε²={kw['epsilon_squared']:.3f} (small) "
    "(disruptive-novel LOWEST → incremental HIGHEST)",
    fontsize=9.5,
)

fig.suptitle(
    "F2 dual-novelty TRENDS — enrichment of the finding that HOLDS (Gate G3 PROCEED; "
    "this does NOT re-gate F2)",
    fontsize=11.5,
)
fig.tight_layout(rect=[0, 0, 1, 0.95])

F2_FIG = FIGURES_DIR / "F2_dual_novelty_trends.png"
fig.savefig(F2_FIG, dpi=DPI)
plt.close(fig)
print("wrote", F2_FIG)

wrote /Users/samersalman/Desktop/SciField/docs/figures/F2_dual_novelty_trends.png


/var/folders/px/cf1lhf6n1w3cdhxtl6k9xglw0000gn/T/ipykernel_65033/3843770547.py:52: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax_imp.boxplot(


In [7]:
# Enforce the < 1 MB figure budget.
sz = F2_FIG.stat().st_size
print(f"F2 trends figure size = {sz / 1024:.1f} KB  (dpi={DPI})")
assert sz < 1_000_000, f"figure too large: {sz} bytes"
print("size assertion PASS — under 1 MB")

F2 trends figure size = 141.5 KB  (dpi=120)
size assertion PASS — under 1 MB


## 7. Provenance sidecar

`record_run` writes the figure sidecar with the full Kruskal-Wallis result, the per-archetype
medians, the temporal-drift endpoints, and the locked column/param choices — so the exact F2-trends
numbers are reproducible and citable by the results-draft + G5 authors.

In [8]:
fig_sidecar = record_run(
    artifact_path=F2_FIG,
    inputs={"archetypes": DATA / "archetypes.parquet"},
    config={
        "figure": "F2_dual_novelty_trends",
        "session": "V1-S15",
        "finding": "F2 dual-novelty (enrichment; does NOT re-gate — Gate G3 PROCEED)",
        "gate_ref": "G3_dual_novelty",
        "dpi": DPI,
        "archetype_col": ARCH_COL,
        "impact_col": IMPACT_COL,
        "n_total_papers": int(n_total),
        "n_null_archetype_excluded": int(n_null_arch),
        "n_labeled_2x2": int(n_labeled),
        "per_archetype_counts": {a: int(df[ARCH_COL].value_counts()[a]) for a in ARCH_ORDER},
        # --- the headline Kruskal-Wallis enrichment ---
        "kruskal_H": float(kw["H"]),
        "kruskal_p_value": float(kw["p_value"]),
        "kruskal_k": int(kw["k"]),
        "kruskal_n": int(kw["n"]),
        "epsilon_squared": float(kw["epsilon_squared"]),
        "median_pctile_by_archetype": {a: float(medians[a]) for a in ARCH_ORDER},
        "n_by_archetype_impact": {a: int(ns[a]) for a in ARCH_ORDER},
        "inversion_reconfirmed": bool(INVERSION_RECONFIRMED),
        # --- temporal drift (G3 endpoints) ---
        "temporal_drift_le2000_ge2020": {a: drift[a] for a in ARCH_ORDER},
        "topn_topics_heatmap": int(TOPN_TOPICS),
        "n_leaf_topics": int(leaf["topic_id"].nunique()),
    },
)
print("recorded figure sidecar:", fig_sidecar)
print("F2 figure exists:", F2_FIG.exists(), "| KB:", f"{F2_FIG.stat().st_size / 1024:.1f}")

recorded figure sidecar: /Users/samersalman/Desktop/SciField/docs/figures/F2_dual_novelty_trends.png.run.json
F2 figure exists: True | KB: 141.5


## 8. Verify artifacts + final F2-trends summary

In [9]:
artifacts = [F2_FIG, F2_FIG.with_suffix(F2_FIG.suffix + ".run.json")]
for a in artifacts:
    ok = a.exists()
    kb = f"{a.stat().st_size / 1024:.1f} KB" if ok else "MISSING"
    print(f"  [{'ok' if ok else 'XX'}] {a.name:<44} {kb}")
    assert ok, f"artifact missing: {a}"
assert F2_FIG.stat().st_size < 1_000_000, "figure exceeds 1 MB"

print("\n" + "=" * 72)
print("F2 dual-novelty TRENDS — enrichment summary (F2 HOLDS; not re-gated)")
print("=" * 72)
print(
    f"Kruskal-Wallis (impact × archetype): H={kw['H']:.2f}, "
    f"p={'<1e-300' if kw['p_value'] == 0.0 else format(kw['p_value'], '.2e')}, "
    f"ε²={kw['epsilon_squared']:.4f}"
)
print("per-archetype median within-year citation percentile:")
for a in ARCH_ORDER:
    print(f"   {a:<26} {medians[a]:.3f}")
print(f"G3 inversion reconfirmed (dn lowest → inc highest, monotone): {INVERSION_RECONFIRMED}")
print(
    f"temporal drift ≤2000→≥2020: conventional-disruptive {CD_DRIFT:+.3f}, "
    f"novel-consolidating {NC_DRIFT:+.3f}"
)
print(f"figure: {F2_FIG}  ({F2_FIG.stat().st_size / 1024:.1f} KB)")

  [ok] F2_dual_novelty_trends.png                   141.5 KB
  [ok] F2_dual_novelty_trends.png.run.json          2.3 KB

F2 dual-novelty TRENDS — enrichment summary (F2 HOLDS; not re-gated)
Kruskal-Wallis (impact × archetype): H=1938.58, p=<1e-300, ε²=0.0237
per-archetype median within-year citation percentile:
   disruptive-novel           0.433
   novel-consolidating        0.502
   conventional-disruptive    0.523
   incremental                0.607
G3 inversion reconfirmed (dn lowest → inc highest, monotone): True
temporal drift ≤2000→≥2020: conventional-disruptive +0.083, novel-consolidating -0.055
figure: /Users/samersalman/Desktop/SciField/docs/figures/F2_dual_novelty_trends.png  (141.5 KB)
